In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../kaggle_prediction_library/') 
import preprocess
import feature_engineering
import submission
import validation
from sklearn.model_selection import train_test_split

# from hyperopt import tpe, fmin, Trials
# import hyperopt.hp as hp

from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import log_loss
from sklearn.feature_selection import chi2
from sklearn.metrics import r2_score
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import brier_score_loss
from sklearn.pipeline import make_pipeline



In [2]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../kaggle_prediction_library/') 
import preprocess
import feature_engineering
import submission
import validation
from sklearn.model_selection import train_test_split

# from hyperopt import tpe, fmin, Trials
# import hyperopt.hp as hp

from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import log_loss
from sklearn.feature_selection import chi2
from sklearn.metrics import r2_score
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import brier_score_loss
from sklearn.pipeline import make_pipeline



In [30]:
to_predict_womens = pd.read_csv("to_predict_women.csv")
to_predict_men = pd.read_csv("to_predict_mens.csv")

In [ ]:
# Define the classifier and parameter grid
model = LogisticRegression(C=0.05)
pipeline = make_pipeline(StandardScaler(), model)
param_grid = {
    'logisticregression__C': [.005, 0.001, .05, 0.01, 0.1],
}

### Evaluate Impact on Overall Model

In [ ]:
to_predict_womens_recent = to_predict_womens[(to_predict_womens.Season >= 2021)
        # filter out first four game
        ]


In [10]:
baseline_features = ['seed_diff', 't1_adj_margin', 't2_adj_margin']
eval_df = validation.run_evaluation_framework(to_predict_womens_recent, pipeline, baseline_features, param_grid, cv_start=2022)

In [11]:
eval_df

,best_params,mean_repeated_cv_score,repeated_cv_confidence_interval,rolling_season_cv
0,{'logisticregression__C': 0.1},-0.148353,"(-0.15133939493055895, -0.14536601553352782)",0.149575


In [122]:
# LATEST
baseline_features = [
    'seed_diff', 
    't1_adj_margin', 't2_adj_margin',
    't1_top8_BPM_weighted_mean', 't2_top8_BPM_weighted_mean',
    # 't1_top8_TO_stdev', 't2_top8_TO_stdev',
    # 't1_top5_PRPG!_median', 't2_top5_PRPG!_median',
    # 't1_top3_DR_median', 't2_top3_DR_median',
    # 't1_top5_STL_cv', 't2_top5_STL_cv',
    # 't1_top3_Min%_median', 't2_top3_Min%_median',
    # 't1_top8_TS_gini', 't2_top8_TS_gini',
    # 't1_top3_USG_gini', 't2_top3_USG_gini',
    ]

eval_df = validation.run_evaluation_framework(to_predict_womens_recent, pipeline, baseline_features, param_grid, cv_start=2022)

eval_df

,best_params,mean_repeated_cv_score,repeated_cv_confidence_interval,rolling_season_cv
0,{'logisticregression__C': 0.1},-0.143741,"(-0.14668144693031104, -0.14080009888286077)",0.145723


In [123]:
# RUN BASELINE USING ALL DATA

# LATEST
baseline_features = [
    'seed_diff', 
    't1_adj_margin', 't2_adj_margin',
    ]

eval_df = validation.run_evaluation_framework(to_predict_womens, pipeline, baseline_features, param_grid, cv_start=2022)

eval_df

,best_params,mean_repeated_cv_score,repeated_cv_confidence_interval,rolling_season_cv
0,{'logisticregression__C': 0.05},-0.14155,"(-0.1432624764824327, -0.13983769450365538)",0.146515


In [ ]:
# -0.155964, 0.160378
# -0.148238, 0.149575
# -0.143946, 0.145723

### Takeaways

Adding the BPM feature seems to help for NCAAW (0.160378 vs. 0.145723). The others don't really. 

However, it's important to note that we only are evaluating on 3 seasons and training on the preceeding ones.

When we train using all data and just the baseline and evaluate those same 3 seasons we get basically a wash (0.145723 vs. 0.146515 for rolling CV, reverse for repeated CV)  

Since training with the mens model and then predicting the womens model seems better (0.1397458542103833), perhaps we should blend them

Train 1 model with all womens data and the baseline features
Train another using mens data to predict womens 
Then average them together.

This would allow us to use the new model but avoid some of the risk.   


In [32]:
to_predict_men_recent = to_predict_men[(to_predict_men.Season >= 2008)]

In [ ]:
baseline_features = ['seed_diff', 't1_adj_margin', 't2_adj_margin']

,seed_diff,t1_adj_margin,t2_adj_margin
320,0,3.968245,-17.174801
321,-13,29.592375,1.918109
322,-15,37.925223,4.919503
323,5,23.549973,20.233231
324,-5,25.184584,13.030481
...,...,...,...
2757,-7,26.636213,11.287927
2758,1,27.364284,27.186115
2759,3,26.577947,29.752956
2760,10,11.287927,27.186115


In [64]:
features = baseline_features + ['t1_top8_BPM_weighted_mean', 't2_top8_BPM_weighted_mean']

In [118]:
features =  ['t1_top8_BPM_weighted_mean', 't2_top8_BPM_weighted_mean',
             't1_adj_margin', 't2_adj_margin']

In [119]:
train = to_predict_men_recent[features + ["Outcome"]]

In [120]:
grid_search = validation.repeated_kfold(train, pipeline, features, "Outcome", param_grid, scoring='neg_brier_score')
pipeline.set_params(**grid_search.best_params_)
pipeline.fit(train[features], train["Outcome"])
y_prob = pipeline.predict_proba(to_predict_womens_recent[features])
loss = brier_score_loss(to_predict_womens_recent["Outcome"].copy(), y_prob[:,1])

loss

0.1397458542103833

In [51]:
loss

0.15280368772459882

In [ ]:
# 0.15280368772459882 -> predict using mens on womens
# 0.13984345298298448 -> with BPM added
# 0.1397458542103833 -> drop seed diff, just efficiencies using mens

### Evaluate Bayesian Approach

In [103]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
import pymc as pm

# Initialize the Bayesian model
class BayesianModel:
    def __init__(self, mens_betas, mens_beta_std, num_features):
        self.mens_betas = mens_betas
        self.mens_beta_std = mens_beta_std
        self.num_features = num_features
        self.trace = None
        self.scaler = StandardScaler()  # Initialize the scaler

    def train(self, X_train, y_train):
        # Ensure X_train is a NumPy array and scale the training data.
        X_train = np.array(X_train)
        X_train_scaled = self.scaler.fit_transform(X_train)
        # Ensure correct dtype for PyMC operations
        X_train_scaled = pm.floatX(X_train_scaled)
        
        with pm.Model() as model:
            # Priors: use men's model coefficients as informative priors.
            beta = pm.Normal("beta", mu=self.mens_betas, sigma=self.mens_beta_std, shape=self.num_features)
            intercept = pm.Normal("intercept", mu=0, sigma=1)  # Less informative prior for intercept

            # Logistic regression likelihood using pm.math.dot for matrix multiplication.
            logits = intercept + pm.math.dot(X_train_scaled, beta)
            p = pm.Deterministic("p", pm.math.sigmoid(logits))

            # Likelihood of the observed data.
            y_obs = pm.Bernoulli("y_obs", p=p, observed=y_train)

            # Sample from the posterior (returns an InferenceData object).
            self.trace = pm.sample(2000, tune=1000, target_accept=0.9)

    def predict(self, X_test):
        if self.trace is None:
            raise ValueError("Model must be trained before making predictions.")

        # Ensure X_test is a NumPy array and scale the test data using the same scaler.
        X_test = np.array(X_test)
        X_test_scaled = self.scaler.transform(X_test)
        
        # Extract posterior samples from the InferenceData object.
        beta_samples = self.trace.posterior["beta"].values  # shape: (chains, draws, num_features)
        intercept_samples = self.trace.posterior["intercept"].values  # shape: (chains, draws)

        # Flatten the chain and draw dimensions.
        beta_samples = beta_samples.reshape(-1, self.num_features)  # shape: (num_samples, num_features)
        intercept_samples = intercept_samples.reshape(-1)  # shape: (num_samples,)

        # Compute logits using numpy.dot.
        # np.dot(X_test_scaled, beta_samples.T) has shape (n_test, num_samples)
        # Adding intercept_samples (shape (num_samples,)) broadcasts as (1, num_samples) and adds to each row.
        logits = np.dot(X_test_scaled, beta_samples.T) + intercept_samples
        # Apply the sigmoid function to obtain probabilities.
        p_samples = 1 / (1 + np.exp(-logits))
        # Average over the posterior samples (axis 1) to produce final probability predictions per test sample.
        p_mean = np.mean(p_samples, axis=1)
        return p_mean


In [107]:
# mens_betas: This is simply the coef_ attribute from the trained logistic regression model. T
# These are the "posterior means" of the model's coefficients based on the men's data.

# mens_beta_std: We use bootstrapping here to estimate the standard deviation of the model’s coefficients. 
# Bootstrapping involves resampling the data with replacement and re-training the model many times (e.g., 1000 iterations). 
# The standard deviation of these coefficients across the different bootstrap samples gives us an estimate of their variability, 
# which we use as the "posterior standard deviations" (mens_beta_std).

from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np

# Assuming `to_predict_men_recent` is your dataset and `features` is the list of feature columns
# Define your features and target
features = baseline_features + ['t1_top8_BPM_weighted_mean', 't2_top8_BPM_weighted_mean']
target = 'Outcome'  # Replace with actual target column name

# Train the logistic regression model on the men's data
model = LogisticRegression(C=0.05)
pipeline_men = make_pipeline(StandardScaler(), model)
pipeline_men.fit(to_predict_men_recent[features], to_predict_men_recent[target])

# Extract the posterior means (betas) from the trained model
mens_betas = pipeline_men.named_steps['logisticregression'].coef_[0]

# Estimate the standard deviation of the coefficients using bootstrapping
n_bootstraps = 1000
bootstrap_betas = []

for _ in range(n_bootstraps):
    # Resample the data with replacement
    X_bootstrap, y_bootstrap = resample(to_predict_men_recent[features], to_predict_men_recent[target])
    pipeline_men.fit(X_bootstrap, y_bootstrap)  # Fit the model on the bootstrapped data
    bootstrap_betas.append(pipeline_men.named_steps['logisticregression'].coef_[0])

# Convert bootstrap results to a numpy array
bootstrap_betas = np.array(bootstrap_betas)

# Compute the standard deviation of the coefficients
mens_beta_std = np.std(bootstrap_betas, axis=0)

print(f"Mens model betas (posterior means): {mens_betas}")
print(f"Mens model beta std (posterior stds): {mens_beta_std}")


Mens model betas (posterior means): [-0.04403922  0.04299076 -0.04299076  0.8903729  -0.8903729 ]
Mens model beta std (posterior stds): [0.08620434 0.08715169 0.08859274 0.08434489 0.08272597]


In [104]:
bayesian_model = BayesianModel(mens_betas, mens_beta_std, num_features=5)

In [105]:
bayesian_model.train(to_predict_womens_recent[features], to_predict_womens_recent["Outcome"])

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta, intercept]


/Users/skylerdale/.virtualenvs/nat_parks/lib/python3.10/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 2 seconds.


In [108]:
# Prepare women's data.
X_women = to_predict_womens_recent[features].values
y_women = to_predict_womens_recent["Outcome"].values

# Set up 5-fold CV on the women's data.
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Lists to store losses for each approach.
losses_men = []    # Approach 1: Men's model (pre-trained on men's data)
losses_women = []  # Approach 2: Women's model trained on CV fold
losses_bayes = []  # Approach 3: Bayesian model

for train_index, test_index in kf.split(X_women):
    # CV split on women's data.
    X_train, X_test = X_women[train_index], X_women[test_index]
    y_train, y_test = y_women[train_index], y_women[test_index]
    
    # --- Approach 1: Men's model predicts on women's test fold.
    y_prob_men = pipeline_men.predict_proba(X_test)
    loss_men = brier_score_loss(y_test, y_prob_men[:, 1])
    losses_men.append(loss_men)
    
    # --- Approach 2: Train women's model on women's training fold.
    pipeline_women = make_pipeline(StandardScaler(), LogisticRegression(C=0.05))
    pipeline_women.fit(X_train, y_train)
    y_prob_women = pipeline_women.predict_proba(X_test)
    loss_women = brier_score_loss(y_test, y_prob_women[:, 1])
    losses_women.append(loss_women)
    
    # --- Approach 3: Bayesian model using men's priors, updated with women's training fold.
    bayesian_model = BayesianModel(mens_betas, mens_beta_std, num_features=len(features))
    bayesian_model.train(X_train, y_train)
    y_prob_bayes = bayesian_model.predict(X_test)
    loss_bayes = brier_score_loss(y_test, y_prob_bayes)
    losses_bayes.append(loss_bayes)

print("Approach 1 (Men's model) average Brier loss:", np.mean(losses_men))
print("Approach 2 (Women's model) average Brier loss:", np.mean(losses_women))
print("Approach 3 (Bayesian model) average Brier loss:", np.mean(losses_bayes))

/Users/skylerdale/.virtualenvs/nat_parks/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta, intercept]


/Users/skylerdale/.virtualenvs/nat_parks/lib/python3.10/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 2 seconds.
/Users/skylerdale/.virtualenvs/nat_parks/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta, intercept]


/Users/skylerdale/.virtualenvs/nat_parks/lib/python3.10/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 2 seconds.
/Users/skylerdale/.virtualenvs/nat_parks/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta, intercept]


...

/Users/skylerdale/.virtualenvs/nat_parks/lib/python3.10/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 2 seconds.
/Users/skylerdale/.virtualenvs/nat_parks/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta, intercept]


/Users/skylerdale/.virtualenvs/nat_parks/lib/python3.10/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 2 seconds.


Approach 1 (Men's model) average Brier loss: 0.1422952850780118
Approach 2 (Women's model) average Brier loss: 0.14454976658606805
Approach 3 (Bayesian model) average Brier loss: 0.14371245322361775
